# 01 — Bronze Layer Ingestion: raw LLM-performance inputs

**Mission**: `6_confirm_LLM_performance_number/missions.md`
**Database**: `/workspace/ICOCO.duckdb`
**Goal**: land every raw JSON / JSONL input used by the LLM-performance verification into the **`bronze`** schema, so that all paper numbers can be re-derived from SQL alone.

| bronze table | source file | expected rows |
|---|---|---|
| `bronze.dirty_ingredient_country_mapping` | `data/GBIF_query/dirty_ingredient_country_mapping.json` | 4,280 |
| `bronze.llm_ingredient` | `data/ingredient_standardize/merged_results.jsonl` | 17,017 |
| `bronze.llm_scientific` | `data/scientific_name/merged_results.jsonl` | 4,280 |

*Why `merged_results.jsonl`?* It is the **raw, per-input** LLM output used by the whole pipeline: one row per `custom_id` with the original `input` and the **three replicas** (`output1/2/3`). Re-loading them through DuckDB's native `read_json_auto` keeps the bronze layer a faithful copy of the source files.

## 0. Setup (connect → close → reconnect)

Only **one** DuckDB connection may be open at a time (see `agents.md`). This notebook
connects, immediately closes (flushes the WAL), then reconnects.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 200
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 1. Bronze schema

Idempotent: the schema is created if missing, and every table below is created with
`CREATE OR REPLACE`, so re-running this notebook never duplicates or mixes data.

In [4]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


### 1.1 `bronze.dirty_ingredient_country_mapping`
Raw mapping *ingredient string → candidate ISO country codes* used by the geographic
pipeline (`old_code/10_geo_vector/GBIF_queryable_rate.ipynb`). Top-level JSON **array** of
objects `{input, country_list[]}`.

In [5]:
%%sql
CREATE OR REPLACE TABLE bronze.dirty_ingredient_country_mapping AS
SELECT
    input::VARCHAR       AS input,
    country_list::VARCHAR[] AS country_list
FROM read_json_auto('/workspace/data/GBIF_query/dirty_ingredient_country_mapping.json', format='array');

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


### 1.2 `bronze.llm_ingredient`
The 3-replica LLM output of the **Ingredient Standardization** task
(Task A in the paper): 17,017 raw ingredient strings → canonical ingredient lists.

In [6]:
%%sql
CREATE OR REPLACE TABLE bronze.llm_ingredient AS
SELECT
    custom_id::VARCHAR AS custom_id,
    input::VARCHAR     AS input,
    output1::VARCHAR   AS output1,
    output2::VARCHAR   AS output2,
    output3::VARCHAR   AS output3
FROM read_json_auto('/workspace/data/ingredient_standardize/merged_results.jsonl', format='newline_delimited');

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


### 1.3 `bronze.llm_scientific`
The 3-replica LLM output of the **Taxonomic Extraction** task
(Task B in the paper): 4,280 ingredient strings → scientific-name candidate lists.

In [7]:
%%sql
CREATE OR REPLACE TABLE bronze.llm_scientific AS
SELECT
    custom_id::VARCHAR AS custom_id,
    input::VARCHAR     AS input,
    output1::VARCHAR   AS output1,
    output2::VARCHAR   AS output2,
    output3::VARCHAR   AS output3
FROM read_json_auto('/workspace/data/scientific_name/merged_results.jsonl', format='newline_delimited');

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


## 2. Integrity checks

Counts must be `4,280 / 17,017 / 4,280`, and a sample of each table must look as expected.

In [8]:
%%sql
SELECT 'bronze.dirty_ingredient_country_mapping' AS tbl, count(*) AS rows
FROM bronze.dirty_ingredient_country_mapping
UNION ALL SELECT 'bronze.llm_ingredient',                count(*) FROM bronze.llm_ingredient
UNION ALL SELECT 'bronze.llm_scientific',                count(*) FROM bronze.llm_scientific;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

tbl,rows
bronze.dirty_ingredient_country_mapping,4280
bronze.llm_ingredient,17017
bronze.llm_scientific,4280


In [9]:
%%sql
SELECT * FROM bronze.dirty_ingredient_country_mapping
WHERE list_has(country_list, '') = false AND len(country_list) > 0
LIMIT 5;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

input,country_list
tangerine juice,"['BE', 'BF', 'BR']"
rattlesnake meat,['US']
scallop,"['MU', 'CN', 'CL', 'BE', 'DK']"
chicken drumettes,"['AS', 'CO', 'BB', 'AU', 'DE', 'BE', 'BM', 'BT']"
onion,"['HR', 'NO', 'GN']"


In [10]:
%%sql
SELECT custom_id, input, output1, output2, output3
FROM bronze.llm_ingredient LIMIT 3;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

custom_id,input,output1,output2,output3
item-00001,-,[],[],[]
item-00002,- alternating strips of peel removed lengthwise and zucchini,"[""zucchini""]","[""zucchini""]","[""zucchini""]"
item-00003,- and,[],"[""and""]",[]


In [11]:
%%sql
SELECT custom_id, input, output1, output2, output3
FROM bronze.llm_scientific LIMIT 3;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

custom_id,input,output1,output2,output3
item-00002,fettuccine,[],[],[]
item-00003,italian seasonings,[],[],[]
item-00004,spelt,"[""Triticum spelta"", ""Triticum aestivum subsp. spelta""]","[""Triticum spelta"", ""Triticum aestivum subsp. spelta""]","[""Triticum spelta"", ""Triticum aestivum subsp. spelta""]"


## Next

Open **`02_LLM_performance_verification.ipynb`**, which reads exclusively from these
`bronze.*` tables to reproduce every LLM-performance number of the paper.